# EfficientAD-S MVTec AD training on Kaggle

This notebook clones the project branch, installs the pinned Kaggle experiment dependencies, validates the attached MVTec AD dataset, and trains category-specific EfficientAD-S models. It is designed to be uploaded and run as one notebook.

MVTec AD remains subject to CC BY-NC-SA 4.0. Keep the attached dataset and generated checkpoints private unless the license and intended use permit distribution.

In [ ]:
# Kaggle setup: enable Internet in Notebook Settings before running this cell.
from pathlib import Path
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/Deepak-kumar-36/Agentic-AI-Powered-Smart-Manufacturing-Quality-Inspection-and-Automation.git'
REPO_BRANCH = 'deepak/feat/local-inspection-api'
REPO_DIR = Path('/kaggle/working/quality-inspection')
DATASET_ROOT = Path('/kaggle/input/datasets/ipythonx/mvtec-ad')
ARTIFACT_ROOT = Path('/kaggle/working/efficientad_s_artifacts')
REFERENCE_REPO_URL = 'https://github.com/nelson1425/EfficientAD.git'
REFERENCE_REPO_DIR = Path('/kaggle/working/reference-efficientad')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repository:', REPO_DIR)
print('Branch:', REPO_BRANCH)
print('Commit:', COMMIT)
if not REFERENCE_REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REFERENCE_REPO_URL, str(REFERENCE_REPO_DIR)], check=True)
TEACHER_PATH = REFERENCE_REPO_DIR / 'models' / 'teacher_small.pth'
if not TEACHER_PATH.is_file():
    raise RuntimeError(f'Reference teacher checkpoint not found: {TEACHER_PATH}')
print('Dataset:', DATASET_ROOT)
print('Reference teacher:', TEACHER_PATH)

In [ ]:
# Install only the Kaggle experiment dependencies.
# Use explicit arguments here so an uploaded notebook remains correct even if the remote branch has not refreshed its requirements file yet.
# Kaggle's current Python 3.13 image requires the torch 2.6/torchvision 0.21 pair.
kaggle_packages = [
    'torch==2.6.0', 'torchvision==0.21.0',
    'numpy==2.1.3', 'Pillow==11.1.0',
    'scipy==1.15.1', 'scikit-learn==1.6.1',
    'requests==2.32.4',
]
# EfficientAD-S does not use Transformers. Remove Kaggle's optional Transformers package so TorchMetrics does not import its incompatible optional text metrics.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'transformers', 'tokenizers'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', *kaggle_packages], check=True)
# Verify the package versions used by the native implementation.
import importlib.metadata as metadata
for package_name in ['torch', 'torchvision', 'numpy', 'scipy', 'requests']:
    try: print(package_name, metadata.version(package_name))
    except metadata.PackageNotFoundError: print(package_name, 'MISSING')
print('Dependencies installed. Restart the Kaggle session now, then rerun the notebook from the first cell so NumPy/SciPy and package metadata are loaded coherently.')

import torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if not torch.cuda.is_available():
    print('WARNING: GPU is unavailable. Training will be much slower on CPU.')
else:
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Run the repository's offline tests before training.
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + test_env.get('PYTHONPATH', '')
test_result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO_DIR / 'tests'), '-v'], cwd=str(REPO_DIR), env=test_env, text=True)
if test_result.returncode != 0:
    raise RuntimeError('Repository tests failed; stop before training.')
print('Repository tests passed.')

## Dataset validation

The dataset path is intentionally fixed to `/kaggle/input/datasets/ipythonx/mvtec-ad`. The reusable project manifest validates image readability, category structure, defect masks, deterministic splits, and file hashes before training.

In [ ]:
from app.efficientad.data import discover_mvtec_manifest

if not DATASET_ROOT.is_dir():
    raise RuntimeError(f'MVTec AD dataset was not found at {DATASET_ROOT}')
if not (DATASET_ROOT / 'bottle' / 'train' / 'good').is_dir():
    raise RuntimeError(f'Invalid MVTec AD root; expected category directories below {DATASET_ROOT}')

CATEGORIES = ['bottle', 'screw', 'capsule']
manifest = discover_mvtec_manifest(DATASET_ROOT, CATEGORIES)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = ARTIFACT_ROOT / 'dataset_manifest.json'
if manifest_path.exists():
    existing_manifest = manifest_path.read_text(encoding='utf-8')
    if existing_manifest != manifest.stable_json() + '\n':
        raise RuntimeError(f'Existing manifest differs from the current dataset: {manifest_path}')
    print('Reusing verified existing manifest:', manifest_path)
else:
    manifest.write(manifest_path)

print('Categories:', manifest.categories)
print('Rows:', len(manifest.rows))
print('Manifest SHA-256:', manifest.digest())
for category in manifest.categories:
    rows = [row for row in manifest.rows if row.category == category]
    print(category, {'train': sum(row.split == 'train' for row in rows), 'validation': sum(row.split == 'validation' for row in rows), 'test': sum(row.split == 'test' for row in rows), 'anomaly_test': sum(row.split == 'test' and row.label == 1 for row in rows)})

## Experiment configuration and provenance

This run uses the native EfficientAD-S implementation in `app.efficientad.native`. The frozen teacher is the cited reference checkpoint from `nelson1425/EfficientAD`; the optional general-image penalty is disabled because no general-image dataset is attached.

In [ ]:
from app.efficientad.config import EfficientADConfig

RUN_MODE = 'smoke'  # change to 'full' for the qualification run
TRAIN_STEPS = 200 if RUN_MODE == 'smoke' else 70000
config = EfficientADConfig(
    model_size='small',
    image_size=(256, 256),
    batch_size=32,
    epochs=100,
    learning_rate=1e-4,
    seed=42,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    implementation='native',
    implementation_version='project-native-efficientad-s-v1',
)
print(json.dumps(config.as_dict(), indent=2))
print('Config SHA-256:', config.digest())

In [ ]:
# Deterministic seeds for the experiment envelope.
import random
import numpy as np
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(config.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Seed configured:', config.seed)

## Category-specific EfficientAD-S training

This cell trains one category-specific native EfficientAD-S model with the cited frozen reference teacher. The general-image penalty is disabled because no general-image dataset is attached. Final test data remains untouched until the qualification evaluation.

In [ ]:
# Re-establish the repository import after a Kaggle kernel restart.
from pathlib import Path
import subprocess
import sys
REPO_URL = 'https://github.com/Deepak-kumar-36/Agentic-AI-Powered-Smart-Manufacturing-Quality-Inspection-and-Automation.git'
REPO_BRANCH = 'deepak/feat/local-inspection-api'
REPO_DIR = Path('/kaggle/working/quality-inspection')
if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from app.efficientad.native import train_category, sha256_file
import torch

REUSE_EXISTING_CHECKPOINTS = True
training_summary = []
for category in CATEGORIES:
    category_root = ARTIFACT_ROOT / 'reference_teacher' / category
    checkpoint = category_root / f'efficientad_s_{category}.pt'
    can_reuse = False
    if REUSE_EXISTING_CHECKPOINTS and checkpoint.is_file():
        try:
            payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
            can_reuse = payload.get('category') == category and payload.get('variant') == 'EfficientAD-S-ReferenceTeacher-MVTecOnly' and int(payload.get('steps', 0)) >= TRAIN_STEPS and payload.get('teacher_sha256') == sha256_file(TEACHER_PATH)
        except Exception:
            can_reuse = False
    if can_reuse:
        training_summary.append({'category': category, 'duration_seconds': 0.0, 'steps': int(payload['steps']), 'checkpoint': str(checkpoint), 'checkpoint_sha256': sha256_file(checkpoint), 'reused_existing': True})
        print(f'Reusing existing checkpoint for {category}: {checkpoint}')
        continue
    started = time.time()
    print(f'\n=== Training {category} ===')
    result = train_category(DATASET_ROOT, category, TEACHER_PATH, category_root, steps=TRAIN_STEPS, device=config.device, seed=config.seed)
    duration = time.time() - started
    training_summary.append({'category': category, 'duration_seconds': duration, 'steps': result.steps, 'checkpoint': str(result.checkpoint), 'checkpoint_sha256': sha256_file(result.checkpoint), 'reused_existing': False})
    print('Finished', category, 'in', round(duration, 2), 'seconds')

print(json.dumps(training_summary, indent=2))

## Validation and untouched-test evaluation

This cell calibrates a high-quantile threshold using validation normals only, then evaluates the untouched test split. It also exports score distributions and a clearly labelled post-hoc test-optimal threshold for review only; that post-hoc value must never be used for qualification. Models are loaded once per category so evaluation measures inference rather than repeated checkpoint construction. Smoke runs produce diagnostic metrics only; full runs are required before qualification. Pixel AUROC is computed only for anomalous test images with verified masks.

In [ ]:
from time import perf_counter
from app.efficientad.native import infer_loaded, load_model_bundle, sha256_file
from PIL import Image
from sklearn.metrics import roc_auc_score
from app.efficientad.evaluation import evaluate_scores

def _rows(category, split):
    return [row for row in manifest.rows if row.category == category and row.split == split]

def _pixel_auc(anomaly_map, mask_path):
    mask = np.asarray(Image.open(DATASET_ROOT / mask_path).convert('L')) > 0
    if anomaly_map.shape != mask.shape:
        raise RuntimeError(f'Heatmap/mask shape mismatch: {anomaly_map.shape} vs {mask.shape}')
    return roc_auc_score(mask.reshape(-1).astype(np.uint8), anomaly_map.reshape(-1))

evaluation_summary = {}
for category in CATEGORIES:
    checkpoint = ARTIFACT_ROOT / 'reference_teacher' / category / f'efficientad_s_{category}.pt'
    validation_rows = _rows(category, 'validation')
    test_rows = _rows(category, 'test')
    validation_scores = []
    test_scores = []
    test_labels = []
    pixel_scores = []
    timings = []
    bundle = load_model_bundle(checkpoint, TEACHER_PATH, config.device)
    for row in validation_rows + test_rows:
        started = perf_counter()
        score, anomaly_map = infer_loaded(bundle, DATASET_ROOT / row.relative_path)
        timings.append(perf_counter() - started)
        if row.split == 'validation':
            validation_scores.append(score)
        else:
            test_scores.append(score)
            test_labels.append(row.label)
            if row.mask_relative_path:
                pixel_scores.append(_pixel_auc(anomaly_map, row.mask_relative_path))
    threshold = float(np.quantile(validation_scores, 0.995))
    report = evaluate_scores(test_scores, test_labels, threshold, 'test')
    posthoc_threshold = select_threshold(test_scores, test_labels)
    posthoc_report = evaluate_scores(test_scores, test_labels, posthoc_threshold, 'test_posthoc_diagnostic')
    quantiles = [0.5, 0.9, 0.95, 0.99, 0.995, 0.999]
    evaluation_summary[category] = {
        'validation_normal_count': len(validation_scores),
        'test_count': len(test_scores),
        'threshold': threshold,
        'image_metrics': report.as_dict(),
        'validation_normal_quantiles': {str(q): float(np.quantile(validation_scores, q)) for q in quantiles},
        'test_normal_quantiles': {str(q): float(np.quantile([s for s, y in zip(test_scores, test_labels) if y == 0], q)) for q in quantiles},
        'test_anomaly_quantiles': {str(q): float(np.quantile([s for s, y in zip(test_scores, test_labels) if y == 1], q)) for q in quantiles},
        'posthoc_test_optimal_threshold_diagnostic': posthoc_report.as_dict(),
        'posthoc_warning': 'Diagnostic only; this threshold used labeled test data and must not be used for qualification.',
        'pixel_auroc_mean': float(np.mean(pixel_scores)) if pixel_scores else None,
        'pixel_auroc_count': len(pixel_scores),
        'latency_ms_mean': float(np.mean(timings) * 1000),
        'latency_ms_p95': float(np.quantile(timings, 0.95) * 1000),
        'checkpoint_sha256': sha256_file(checkpoint),
    }
print(json.dumps(evaluation_summary, indent=2, default=str))

## Export run metadata

This metadata is deliberately conservative. It records the run environment and hashes the dataset/configuration. Replace the qualification status only after reviewing actual validation thresholds, untouched test metrics, checkpoint hashes, and artifact completeness.

In [ ]:
run_metadata = {
    'experiment': 'efficientad-s-reference-teacher-mvtec-only-v1',
    'run_mode': RUN_MODE,
    'repository_url': REPO_URL,
    'repository_branch': REPO_BRANCH,
    'repository_commit': COMMIT,
    'dataset_root': str(DATASET_ROOT),
    'dataset_manifest_sha256': manifest.digest(),
    'config': config.as_dict(),
    'config_sha256': config.digest(),
    'implementation': 'native',
    'implementation_version': 'project-native-efficientad-s-v1',
    'categories': CATEGORIES,
    'license': 'CC BY-NC-SA 4.0',
    'teacher_source': REFERENCE_REPO_URL,
    'teacher_checkpoint': str(TEACHER_PATH),
    'teacher_sha256': sha256_file(TEACHER_PATH),
    'general_image_penalty': False,
    'python': sys.version,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'training_summary': training_summary,
    'evaluation_summary': evaluation_summary,
    'qualification_status': 'NOT_QUALIFIED_UNTIL_METRICS_AND_ARTIFACT_HASHES_ARE_REVIEWED'
}
metadata_path = ARTIFACT_ROOT / 'run_metadata.json'
metadata_path.write_text(json.dumps(run_metadata, indent=2, default=str) + '\n', encoding='utf-8')
print(json.dumps(run_metadata, indent=2, default=str))
print('Artifacts:', ARTIFACT_ROOT)

## Download outputs

After the run completes, use Kaggle's file browser to download `/kaggle/working/efficientad_s_artifacts/`. Place the downloaded category artifacts under the local project's configured model-artifact directory only after verifying the manifest and hashes. Do not commit checkpoints, datasets, or generated outputs to GitHub.

In [ ]:
# Reload one trained checkpoint and export score, heatmap, and overlay evidence.
# This cell is restart-safe and can bootstrap imports if it is run manually.
from pathlib import Path
import os
import subprocess
import sys
import numpy as np
REPO_URL = 'https://github.com/Deepak-kumar-36/Agentic-AI-Powered-Smart-Manufacturing-Quality-Inspection-and-Automation.git'
REPO_BRANCH = 'deepak/feat/local-inspection-api'
REPO_DIR = Path('/kaggle/working/quality-inspection')
DATASET_ROOT = Path('/kaggle/input/datasets/ipythonx/mvtec-ad')
ARTIFACT_ROOT = Path('/kaggle/working/efficientad_s_artifacts')
REFERENCE_REPO_URL = 'https://github.com/nelson1425/EfficientAD.git'
REFERENCE_REPO_DIR = Path('/kaggle/working/reference-efficientad')
if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if not REFERENCE_REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REFERENCE_REPO_URL, str(REFERENCE_REPO_DIR)], check=True)
TEACHER_PATH = REFERENCE_REPO_DIR / 'models' / 'teacher_small.pth'
if not TEACHER_PATH.is_file():
    raise RuntimeError(f'Reference teacher checkpoint not found: {TEACHER_PATH}')
CATEGORIES = ['bottle', 'screw', 'capsule']
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
sample_image = next((DATASET_ROOT / CATEGORIES[0] / 'test').glob('*/*.png'))
sample_checkpoint = ARTIFACT_ROOT / 'reference_teacher' / CATEGORIES[0] / f'efficientad_s_{CATEGORIES[0]}.pt'
if not sample_checkpoint.is_file():
    raise RuntimeError(f'Trained checkpoint not found: {sample_checkpoint}. Run the training cells first.')
from app.efficientad.native import infer_checkpoint, overlay_image
score, anomaly_map = infer_checkpoint(sample_checkpoint, TEACHER_PATH, sample_image, device)
evidence_dir = ARTIFACT_ROOT / 'evidence'
evidence_dir.mkdir(parents=True, exist_ok=True)
np.save(evidence_dir / 'sample_heatmap.npy', anomaly_map)
overlay_image(sample_image, anomaly_map / max(float(anomaly_map.max()), 1e-6), evidence_dir / 'sample_overlay.png')
print({'image': str(sample_image), 'score': score, 'heatmap': str(evidence_dir / 'sample_heatmap.npy'), 'overlay': str(evidence_dir / 'sample_overlay.png')})